<a href="https://colab.research.google.com/github/avikumart/DA-DS-Questions/blob/main/R_codes/web_scraping_and_shiny_app_using_R_hw4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
library(rvest)
library(tidyverse)
library(lubridate)
library(xml2)

In [25]:
# scrape the covid data
url.deaths <- "https://en.wikipedia.org/wiki/COVID-19_pandemic_death_rates_by_country"

In [35]:
url.vaccines <- "https://en.wikipedia.org/wiki/COVID-19_vaccine"

In [32]:
# write the scraping function for the the both sites
page <- read_html(url.deaths)

tbl_node <- page %>%
  html_node(xpath = "//table[contains(@class, 'wikitable') and contains(., 'Deaths / million')]")

tbl <- tbl_node %>%
  html_table(fill = TRUE)

In [33]:
colnames(tbl) <- c("Country","Deaths_per_million","Deaths","Cases")

tbl_clean <- tbl %>%
  mutate(
  Deaths_per_million = as.numeric(gsub(",", "", Deaths_per_million)),
  Deaths = as.numeric(gsub(",", "", Deaths)),
  Cases = as.numeric(gsub(",", "", Cases))
  )

head(tbl_clean)

Warning message:
“There were 3 warnings in `mutate()`.
The first warning was:
ℹ In argument: `Deaths_per_million = as.numeric(gsub(",", "",
  Deaths_per_million))`.
Caused by warning:
! NAs introduced by coercion
ℹ Run `dplyr::last_dplyr_warnings()` to see the 2 remaining warnings.”


Country,Deaths_per_million,Deaths,Cases
<chr>,<dbl>,<dbl>,<dbl>
World[a],892,7101682,778602438
Peru,6603,221060,4532724
Bulgaria,5679,38767,1339176
North Macedonia,5429,9991,352093
Bosnia and Herzegovina,5119,16406,404289
Hungary,5072,49124,2238461


In [34]:
tbl_clean

Country,Deaths_per_million,Deaths,Cases
<chr>,<dbl>,<dbl>,<dbl>
World[a],892,7101682,778602438
Peru,6603,221060,4532724
Bulgaria,5679,38767,1339176
North Macedonia,5429,9991,352093
Bosnia and Herzegovina,5119,16406,404289
Hungary,5072,49124,2238461
Croatia,4807,18784,1360993
Slovenia,4686,9914,1363849
Georgia,4519,17151,1864386


In [36]:
# scrape the data for the vaccines data
page1 <- read_html(url.vaccines)

tbl_node1 <- page1 %>%
  html_node(xpath = '//*[@id="mw-content-text"]/div[1]/table[4]')

tbl1 <- tbl_node1 %>%
  html_table(fill = TRUE)

In [37]:
head(tbl1)

Common name,Type (technology),Country of origin,First authorization,Notes
<chr>,<chr>,<chr>,<chr>,<chr>
Authorized in more than 10 countries and by the WHO,Authorized in more than 10 countries and by the WHO,Authorized in more than 10 countries and by the WHO,Authorized in more than 10 countries and by the WHO,Authorized in more than 10 countries and by the WHO
Oxford–AstraZeneca,Adenovirus vector,"United Kingdom,Sweden",December 2020,
Pfizer–BioNTech,RNA,"Germany,United States",December 2020,Both original and Omicron variant versions
Janssen (Johnson & Johnson),Adenovirus vector,"United States,Netherlands",February 2021,
Moderna,RNA,United States,December 2020,Both original and Omicron variant versions
Sinopharm BIBP,Inactivated,China,July 2020,


In [38]:
# map visualization of death rates by each country
install.packages(c("rvest", "dplyr", "rnaturalearth",
                   "rnaturalearthdata", "sf", "ggplot2"))

library(rvest)
library(dplyr)
library(ggplot2)
library(rnaturalearth)
library(rnaturalearthdata)
library(sf)

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘proxy’, ‘e1071’, ‘wk’, ‘terra’, ‘classInt’, ‘s2’, ‘units’



Attaching package: ‘rnaturalearthdata’


The following object is masked from ‘package:rnaturalearth’:

    countries110


Linking to GEOS 3.12.1, GDAL 3.8.4, PROJ 9.3.1; sf_use_s2() is TRUE



In [39]:
world <- ne_countries(scale = "medium", returnclass = "sf")

In [42]:
tbl_clean <- tbl_clean %>%
  rename("sovereignt" = "Country")

In [44]:
world_data <- left_join(world, tbl_clean, by = c("name" = "sovereignt"))

## Shiny practice app

In [20]:
ui <- fluidPage(

  # App title ----
  titlePanel("Tabsets"),

  # Sidebar layout with input and output definitions ----
  sidebarLayout(

    # Sidebar panel for inputs ----
    sidebarPanel(

      # Input: Select the random distribution type ----
      radioButtons("dist", "Distribution type:",
                   c("Normal" = "norm",
                     "Uniform" = "unif",
                     "Log-normal" = "lnorm",
                     "Exponential" = "exp")),

      # Input: Slider for the number of observations to generate ----
      sliderInput("n",
                  "Number of observations:",
                  value = 500,
                  min = 1,
                  max = 1000)

    ),

    # Main panel for displaying outputs ----
    mainPanel(plotOutput("plot"))
  )
)

# Define server logic for random distribution app ----
server <- function(input, output) {

  # Generate a plot of the data ----
  # Also uses the inputs to build the plot label. Note that all expressions are called in the sequence
  # implied by the dependency graph.
   output$plot <- renderPlot({
    n <- input$n
    dist <- switch(input$dist,
                   norm = rnorm,
                   unif = runif,
                   lnorm = rlnorm,
                   exp = rexp,
                   rnorm)

    hist(dist(input$n), main="Histogram")
  })
}

# Create Shiny app ----
shinyApp(ui, server)

ERROR: Error in fluidPage(titlePanel("Tabsets"), sidebarLayout(sidebarPanel(radioButtons("dist", : could not find function "fluidPage"
